# Raw soil-temperature profile `TS_FF1` (2020-2025) from database (influxdb)

> ## 🛑 Retired - this notebook is no longer run
>
> It was written to serve the **high-resolution** cross-check in the per-sensor `TS` screening notebooks: those notebooks compared their target against the other twelve channels of the profile at 1MIN/10MIN resolution, and this file existed so that each of them did not have to download 2.5 million rows x 13 fields again.
>
> **That cross-check was moved out of screening.** The `TS` screening notebooks were simplified to do what the diive template does - remove the worst outliers on the high-res record and resample to 30MIN - because at raw resolution the comparison costs far more than it returns. The profile comparison, the depth ordering of amplitude and phase, and the sensor-generation questions are now settled in `30_PRODUCTS/`, **on the 30MIN data**. Something similar to this notebook will be rebuilt there from the half-hourly record.
>
> Nothing reads `TS_FF1_PROFILE_RAW_2020-2025.parquet` any more. The file itself is still in the external data folder and can be deleted.
>
> Kept, unrun, because the sections below measure things about the raw record that are expensive to rediscover - what a failed SDI-12 read looks like on `TS`, the two time resolutions, the 0.05 degC quantisation of the 1MIN era, and the per-channel coverage. Read them; do not run them.

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `TS_FF1_*`, all thirteen channels of the FF1 soil profile (`0.05` m ×4, `0.1` m ×2, `0.2` m ×2, `0.3` m ×2, `0.5` m ×2, `0.6` m ×1) &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2020-2025  
**Produces**: `TS_FF1_PROFILE_RAW_2020-2025.parquet` — the raw profile, values only, on local `TIMESTAMP_END`  
**Consumed by**: nothing - see the retirement note above. It was the cross-check reference of the per-sensor `TS` screening notebooks  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download the **raw** `TS_FF1` soil-temperature profile — all thirteen fields at once — from the InfluxDB database and write it to a single parquet file. Nothing is screened, corrected, resampled or uploaded here. This notebook is pure I/O. It is the soil-temperature twin of `SWC/SWC_FF1_PROFILE_2020-2025.ipynb` and the one sanctioned file output of the `TS/` folder.

**Why it exists.** Every per-sensor screening notebook in this folder cross-checks its target against the rest of the profile — for soil temperature that check is sharper than for soil moisture, because the diurnal amplitude must damp and the daily maximum must lag monotonically with depth, so a probe that has moved or been exposed breaks the ordering visibly. Doing that check needs the *other* twelve channels. Without this file each notebook would download the whole profile again: **2 529 202 rows × 13 fields**, several minutes of transfer, for byte-identical data. This notebook downloads it **once**. Each screening notebook then reads its companions from here and downloads only **its own target** live from the database.

**Flow:** connect → download the thirteen raw fields → check the frame → write parquet → read the file back and verify.

> **Database access** needs the `influxdb-client` package, which this project pulls in via the **`diive[db]` extra** (declared in `pyproject.toml`). diive *also* ships a `db` dependency group, but dependency groups are local to the project that declares them — `uv sync --group db` only works inside the diive repo, not from here. From this repo the extra is the only route.

### 💡 Values only, on purpose — this file carries no database tags
`InfluxIO.download` returns two views of the same query: `data_simple`, a plain DataFrame with one column per field, and `data_detailed`, a dict `{field: DataFrame}` in which every field carries **its database tags** (`units`, `gain`, `offset`, `hpos`/`vpos`, `data_version`, …). **This notebook saves `data_simple` — the values, without the tags.**

This is the single most important thing to understand about the file, so it is worth stating plainly:

- **What the file is for.** It serves the **companion** channels in the per-sensor screening notebooks, and those are used for one purpose only: the **30MIN cross-check** — half-hourly means of the other channels, compared against the channel being screened. A mean needs values and timestamps. Nothing in that comparison reads a tag.
- **What the file is *not* for.** It cannot stand in for a screening notebook's own download. `StepwiseMeteoScreeningDb` is constructed from `data_detailed` and **needs the tags**: they travel with the series through screening and are written back to the database on upload. A tagless frame would silently lose them.

So each screening notebook still downloads **its own target variable live from the database**, with tags, and reads only its **twelve companions** from this file. That turns thirteen full-profile downloads into one full-profile download plus thirteen single-variable ones.

### ⚠️ This file is raw and unscreened — read this before using it
It is the raw database record exactly as stored, so every artefact is still in it. The list below is not generic caution: each item was measured on the file this notebook writes, on **22 Jul 2026**, and the numbers are the measured ones. A consumer that treats this file as clean data will be wrong in these specific ways.

**1. A failed SDI-12 read is a number, not a gap — and for `TS` that number is `0.0`.** The CR1000 firmware predates SDI-12 NaN support (a deliberate decision logged on 2020-04-10), so a failed read lands in the record as the sensor's conversion evaluated at zero response. For soil *moisture* that is a fixed negative value per sensor and therefore easy to catch; for soil *temperature* it is **exactly `0.00 °C`**, which is a physically possible soil temperature. Measured over 2020-2025 there are only five such timestamps, and they are unambiguous because **all eleven SDI-12 channels go to `0.0` in the same minute** while the two analogue 109 thermistors carry on:

| timestamp (local, `TIMESTAMP_END`) | affected | context |
|---|---|---|
| 2024-04-05 01:23 | `TS_FF1_0.6_1` only | during the April 2024 SDI-12 bus failure |
| 2025-12-18 17:45 | all eleven SDI-12 channels | logger-program upload |
| 2025-12-19 09:57 | all eleven | logger-program upload |
| 2025-12-19 10:31 | all eleven | logger-program upload |
| 2025-12-19 10:39 | all eleven | logger-program upload |

At 2025-12-18 17:45 the whole profile reads `0.0`; one minute later it reads 5.9-7.5 °C and stays there. `notna()` therefore does **not** mean "measured". The two remote logger-program uploads of 18/19 Dec 2025 are in the fieldbook and explain all four December timestamps.

**2. The April 2024 SDI-12 bus failure also produces garbage *high* values.** The 0.3 m TEROS 12 failed on 2024-04-03 and dragged down every sensor on the same port. Between 2024-04-03 and 2024-04-09 the eleven SDI-12 channels return 41 records above 25 °C, in this pattern:

| field | records > 25 °C | values |
|---|---|---|
| `TS_FF1_0.05_1` | 2 | 40.5, 41.0 |
| `TS_FF1_0.05_2` | 6 | 54.0 … 80.0 |
| `TS_FF1_0.1_1` | 3 | 40.5, 40.6 |
| `TS_FF1_0.1_2` | 7 | 40.5 … 202.5 |
| `TS_FF1_0.2_1` | 1 | 53.35 |
| `TS_FF1_0.2_2` | 8 | 112.05 … 121.7 |
| `TS_FF1_0.5_2` | 3 | 146.95 … 277.0 |
| `TS_FF1_0.6_1` | 11 | 303.99 … 307.56 |

`TS_FF1_0.3_1`, `TS_FF1_0.3_2` and `TS_FF1_0.5_1` produce none. So the garbage is not a fixed per-sensor sentinel like the `0.0` above — it is decode noise, different every time, and only an absolute-limits test catches it. **Never clip to the limit**: clipping would turn a 307 °C decode error into a fabricated 30 °C.

**3. `0.3` holds two different sensors under one field name — but only in `_2`.** `TS_FF1_0.3_2` is the temperature channel of the TEROS 12 that failed in April 2024, was unplugged on 2024-04-23 and replaced on 2025-03-04 about **40 cm downslope**. It is empty from **2024-04-12 to 2025-03-03 (326 days)** and holds **2 058 796** records against ~2.50 M for its neighbours. Unlike soil moisture, where the relocation moved the level by ~3 % VWC, the temperature barely noticed: the daily-mean difference `0.3_2 − 0.3_1` is **−0.137 K** over 2023-03-05…2024-04-10 and **−0.087 K** over 2025-03-05…2025-12-31, i.e. the relocation changed the pair offset by 0.05 K. That is reassuring but it is **not** a licence to treat the two eras as one sensor: a companion mean that simply averages "whatever is present" still changes membership mid-record at 2024-04-12 and again at 2025-03-04. Decide deliberately what to do about that; do not let it happen by accident.

**4. Two raw time resolutions.** 10MIN until the March 2021 logger rebuild, 1MIN after, plus five stray 2-minute records. The frequency-group report below makes them visible.

**5. `TS_FF1_0.05_4` has a real fault window that is not a gap.** Between **2022-02-24 and 2022-05-12** this 109 thermistor records **67 days with a daily amplitude above 5 K**, median daily amplitude **22.36 K in March 2022** against **1.30 K in March 2023**, running from **−5.49 °C** to **+31.53 °C**. Its twin `TS_FF1_0.05_3`, 5 cm deep a few metres away, stays between 1.8 and 9 °C with 0.5-1.3 K amplitude through the same weeks. A quieter repeat follows the 23 Oct 2025 windstorm: median daily amplitude 3.38 K in Nov 2025 and 1.86 K in Dec 2025, against 0.37-0.71 K in every earlier Nov/Dec. Outside those windows `0.05_4` runs at 1.0-1.7 K summer amplitude against 0.9-1.2 K for `0.05_3` — so the open fieldbook concern of 2026-07-17 ("too close to the surface? Very strong daily cycle") is **not** supported by the normal record; what the record does show is two episodes of a sensor that was, at least temporarily, not in the soil. That is a hypothesis for the `0.05_4` screening notebook to settle, not a conclusion asserted here.

Screening decisions belong in the per-sensor notebooks, not here. This notebook removes nothing.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the *end* of the averaging interval).

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). `InfluxIO.download` applies it on download, so what comes back — and what this notebook writes — is **local time (UTC+1), timezone-naive, `TIMESTAMP_END`**.

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| **This parquet file** | **local (UTC+N)** | **`TIMESTAMP_END`** | this notebook |
| In the screening notebooks | local | `TIMESTAMP_END`, resampled to 30MIN means | consumer |

Nothing is converted to `TIMESTAMP_MID` here — that conversion happens inside `StepwiseMeteoScreeningDb`, on the target variable, in the screening notebooks. The cross-check those notebooks run works on `TIMESTAMP_END` as downloaded, which is exactly what this file holds, so consumer and file agree with no shift in between. (The air-temperature reference a `TS` notebook may pull from `30_PRODUCTS/02_METEO_TA_GAPFILLED_2004-2025.parquet` is the exception: that file is stored on `TIMESTAMP_MID` and must be shifted **+15 min** before it is used beside anything from here.) `TIMEZONE_OFFSET_TO_UTC_HOURS` **must match the value used in the screening notebooks** (`1`); if it ever changes, this file has to be rewritten with it.

## 🔁 When this notebook must be re-run
Re-run it — **before** the screening notebooks — whenever

- the raw record is **extended** (a new year of data arrives), or
- the raw data is **re-ingested or corrected** in the database.

If it is not re-run, the screening notebooks cross-check against a **stale profile**: the companion channels simply stop before the screened period ends, and the comparison quietly covers less than it claims. The screening notebooks therefore *assert* that this file covers the period they screen, so a stale file **fails loudly** instead of silently. That assertion is the safety net; re-running this notebook is the fix.

Re-running is idempotent — the parquet is overwritten in place. When the period grows, update `STOP` and the year range in `OUTNAME` (and rename this notebook to match), then update the readers.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site and variables**
- `SITE`: site ID, used to build the bucket name.
- `PROFILE`: the horizontal position tag of the soil profile (`FF1`).
- `FIELDS`: the thirteen InfluxDB `_field` names, written out **explicitly**. They are deliberately *not* built from a `depths × replicates` comprehension, because the profile is not a regular grid: four sensors sit at 0.05 m, two each at 0.1/0.2/0.3/0.5 m and one at 0.6 m, and three different sensor generations share the measurement. A comprehension would hide exactly the thing a reader needs to see. The comments name which sensor each channel belongs to.
- `REQUIRED_FIELDS`: the subset that **must** carry data for this download to be considered good — see *The all-NaN check* below.
- `MEASUREMENT`: exactly **one** measurement grouping the variables — `TS` for soil temperature.

**Time range to download**
- `START`: first timestamp — **is** included.
- `STOP`: upper bound — **is not** included. Mirrors the screening notebooks exactly, so the file always covers what they screen.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the critical timestamp knob — see *Timestamp convention* above. Must be the same value the screening notebooks use.
- `DATA_VERSION`: the source data version in the database (`raw`). This notebook deals only with raw data; the screened versions live in the processed bucket and are written by the per-sensor notebooks.
- `DIRCONF`: local folder holding the database connection config.

**Output**
- `OUTDIR` / `OUTNAME` / `OUTPATH`: the parquet written by this notebook. It lives in the external (untracked) data folder under **the same relative path as this notebook**, and `OUTNAME` starts with this notebook's name, so code and data line up by eye.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
PROFILE = 'FF1'  # horizontal position (soil profile)
MEASUREMENT = 'TS'

# --- Variables: the thirteen TS channels of the FF1 profile ---
# Three sensor generations share the measurement TS and they are NOT a regular grid, so the list
# is written out by hand. Which channel belongs to which sensor was established from the data
# (see 'Which channel is which sensor' below), not assumed from the name:
#
#   _2  METER TEROS 12, installed 19 Mar 2020, five probes at 0.05/0.1/0.2/0.3/0.5 m. These are the
#       *same physical probes* as SWC_FF1_<depth>_1 - one device reporting VWC, EC and TS.
#   _1  the water-potential sensors of the same install: 3x METER TEROS 21 + 3x MPS-2, six channels
#       at 0.05/0.1/0.2/0.3/0.5/0.6 m. Which three are TEROS 21 and which three are MPS-2 is not
#       resolvable from the GIN export - it lists them under one collective entry.
#   _3/_4  Campbell 109 thermistors, added 2021-03-24 at the logger rebuild, both at 0.05 m, installed
#       beside the soil heat flux plates G_FF1_0.05_1 and G_FF1_0.05_2 respectively. They serve the
#       soil-heat-flux storage term. Analogue, not SDI-12, and on a different port - which is why they
#       are the only channels that survived the April 2024 SDI-12 bus failure.
FIELDS = [
    'TS_FF1_0.05_1',  # TEROS 21 / MPS-2 (water potential sensor), 0.05 m
    'TS_FF1_0.05_2',  # TEROS 12 (= SWC_FF1_0.05_1), 0.05 m
    'TS_FF1_0.05_3',  # Campbell 109 beside heat flux plate G_FF1_0.05_1, from 2021-03-26
    'TS_FF1_0.05_4',  # Campbell 109 beside heat flux plate G_FF1_0.05_2, from 2021-03-26
    'TS_FF1_0.1_1',   # TEROS 21 / MPS-2, 0.1 m
    'TS_FF1_0.1_2',   # TEROS 12 (= SWC_FF1_0.1_1), 0.1 m
    'TS_FF1_0.2_1',   # TEROS 21 / MPS-2, 0.2 m  <- the channel named in the 2025-06-27 INCIDENT
    'TS_FF1_0.2_2',   # TEROS 12 (= SWC_FF1_0.2_1), 0.2 m
    'TS_FF1_0.3_1',   # TEROS 21 / MPS-2, 0.3 m
    'TS_FF1_0.3_2',   # TEROS 12 (= SWC_FF1_0.3_1), 0.3 m  <- two sensors, dead 2024-04-12..2025-03-03
    'TS_FF1_0.5_1',   # TEROS 21 / MPS-2, 0.5 m
    'TS_FF1_0.5_2',   # TEROS 12 (= SWC_FF1_0.5_1), 0.5 m
    'TS_FF1_0.6_1',   # TEROS 21 / MPS-2, 0.6 m - the only channel at this depth, no TEROS 12 here
]

# The eleven SDI-12 channels of the March 2020 install. They exist for the whole period this notebook
# covers, so an empty one means the download failed and nothing may be written. The two Campbell 109
# channels are deliberately NOT in this list - see 'The all-NaN check' below.
REQUIRED_FIELDS = [
    'TS_FF1_0.05_1', 'TS_FF1_0.05_2',
    'TS_FF1_0.1_1', 'TS_FF1_0.1_2',
    'TS_FF1_0.2_1', 'TS_FF1_0.2_2',
    'TS_FF1_0.3_1', 'TS_FF1_0.3_2',
    'TS_FF1_0.5_1', 'TS_FF1_0.5_2',
    'TS_FF1_0.6_1',
]

# --- Time range to download ---
# Raw resolution is 10MIN until the March 2021 logger rebuild and 1MIN after, so this range is
# ~2.53 million rows over 13 fields. The profile starts 2020-04-10 15:00; asking for 2020-01-01
# simply starts at the first record.
# Keep these identical to the screening notebooks - they assert that this file covers the period
# they screen.
START = '2020-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Output ---
# Data files live in the external (untracked) data folder, under the same relative path as this
# notebook. OUTNAME is this notebook's name plus what the file holds.
OUTDIR = (r'F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data'
          r'\workflow\10_METEO\20_SCREENING\TS')
OUTNAME = 'TS_FF1_PROFILE_RAW_2020-2025'

### 🔬 Which channel is which sensor — and how that was established
The field names do not say which physical device a channel belongs to, and getting it wrong would put the wrong companion in every cross-check. Three measurements settle it, all made on this file on 22 Jul 2026:

1. **Sensor type, from the value grid.** The eleven `_1`/`_2` channels are quantised: 99.54-99.70 % of their values fall exactly on a 0.05 °C grid. The two `_3`/`_4` channels are continuous — 0.007-0.008 % happen to land on that grid, which is what you get by chance. So `_3`/`_4` are the analogue Campbell 109 thermistors and the other eleven are SDI-12.
2. **Count, from the fieldbook.** The 19 Mar 2020 install was "5 Teros12 sensors … 3 Teros21 sensors and 3 MPS old sensors" — eleven SDI-12 devices, and eleven SDI-12 temperature channels exist. `_2` exists only at the five TEROS 12 depths (0.05/0.1/0.2/0.3/0.5); `_1` exists at six depths including 0.6 m, where there is no TEROS 12. Six water-potential sensors, six `_1` channels.
3. **The decisive one: the 0.3 m hole.** The TEROS 12 at 0.3 m was unplugged on 2024-04-23 and replaced on 2025-03-04. Its soil-moisture channel `SWC_FF1_0.3_1` therefore has a known 326-day hole. Comparing the `notna()` pattern of both 0.3 m temperature channels against it: `TS_FF1_0.3_2` agrees on **100.000 %** of shared rows, `TS_FF1_0.3_1` on 81.891 %. **`_2` is the TEROS 12.**

This matters immediately for one fieldbook entry. The INCIDENT of 2025-06-27 is filed under the 0.2 m TEROS 12 and reads "TS_FF1_02_1 does not work regularly since 27.02". In the data, `TS_FF1_0.2_1` drops from 1 440 records/day to between 18 and 138 records/day on **2025-06-26 … 2025-07-06**, while `TS_FF1_0.2_2` — the TEROS 12's own temperature channel — stays at a full 1 440/day throughout. So the entry names the right *field* but the wrong *device*, and its "since 27.02" is not visible in the record at all: February through 25 June 2025 is complete. Evidence first, fieldbook second.

## 🤖 Auto settings

### Buckets and output path (do not adjust)

In [ ]:
from pathlib import Path

BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
OUTPATH = Path(OUTDIR) / f'{OUTNAME}.parquet'

assert all(f in FIELDS for f in REQUIRED_FIELDS), (
    f'(!) REQUIRED_FIELDS not a subset of FIELDS: {[f for f in REQUIRED_FIELDS if f not in FIELDS]}')
assert len(set(FIELDS)) == len(FIELDS), '(!) duplicate entries in FIELDS'

print(f'Downloading:  {len(FIELDS)} fields')
for _f in FIELDS:
    print(f'                {_f}{"   (required)" if _f in REQUIRED_FIELDS else ""}')
print(f'From bucket:  {BUCKET_RAW}  (measurement {MEASUREMENT}, data version {DATA_VERSION})')
print(f'Period:       {START}  ->  {STOP}  (stop not included)')
print(f'Writing to:   {OUTPATH}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd

from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra
from diive.core.times.times import detect_freq_groups  # the same grouping the screening uses internally

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## 🧰 Helper: the validator every check goes through
One function states what a valid profile frame is, and it is used three times: on what the download returned, before writing, and on what was read back from the written file. Everything a consumer of this file silently assumes is checked in one place — index name, index type, timezone, ordering, duplicates, and that all thirteen channels are present.

It raises rather than returning `False`, so a bad frame cannot slip past an ignored return value, and it returns the frame so it can be used inline.

### 🤔 The all-NaN check: where this deliberately differs from the SWC twin
`SWC/SWC_FF1_PROFILE_2020-2025.ipynb` **raises** if *any* requested column holds not a single value, on the argument that a column of NaN is the one failure mode that looks like valid data to every downstream mean. That argument is right, and it is right there: the five SWC depths are one install, one start date, one lifetime. Empty means broken.

Here it is wrong as a *fatal* rule, for a reason specific to `TS`: the thirteen channels are **three sensor generations with different start dates**. `TS_FF1_0.05_3` and `TS_FF1_0.05_4` first appear on **2021-03-26 15:22**, at the logger rebuild — 350 days after the rest of the profile. Over the period this notebook downloads they are far from empty (2 479 116 records each), but for any window that ends before the rebuild they are **legitimately empty**, and a validator that refuses to write the file in that case would be refusing over a fact of site history. That is the kind of check people delete rather than fix, and a deleted check protects nothing.

So the check is split rather than dropped:

- **Reported, loudly, for every field.** Any column without a single value is printed as a `(!) REPORT` line naming it. Emptiness never happens silently.
- **Fatal only for `REQUIRED_FIELDS`** — the eleven SDI-12 channels of the March 2020 install, which exist across the entire period requested here. If one of *those* comes back empty, the download failed and nothing is written.

The alternative — keep the blanket raise and drop the two 109 channels from `FIELDS` — was rejected because nothing is dead: measured on 22 Jul 2026, **all thirteen fields carry data** over 2020-2025, the thinnest being `TS_FF1_0.3_2` at 2 058 796 records (81.40 % coverage, the 2024/25 sensor swap). Dropping live channels to satisfy a check would remove the two channels that are *most* useful as companions, since the 109s are the only ones that kept logging through the April 2024 SDI-12 bus failure.

`required=None` keeps the strict SWC behaviour (every field required), so the default is the safe one and the relaxation has to be asked for by name.

In [ ]:
def check_profile_frame(df: pd.DataFrame, fields: list, required: list = None,
                        verbose: bool = True) -> pd.DataFrame:
    """Validate a raw soil-profile frame; raise ValueError if anything is off.

    Checks exactly what a reader of the exported parquet file assumes to be true:
    a DatetimeIndex named TIMESTAMP_END, timezone-naive (InfluxIO already shifted
    the UTC stamps to local time, so a tz-aware index would mean the shift was
    applied twice or not at all), sorted, without duplicate timestamps, and one
    column per requested field.

    Emptiness is handled in two tiers, see the markdown cell above:
      - every field without a single value is REPORTED,
      - a field listed in *required* that is empty RAISES.
    *required* defaults to *fields*, i.e. to the strict all-or-nothing behaviour.

    Note that the column ORDER of the returned frame is the database's, not the
    order of *fields* - readers must select companions by name, never by position.

    Returns *df* unchanged, so it can be used inline.
    """
    if not isinstance(df, pd.DataFrame):
        raise ValueError(f'expected a DataFrame, got {type(df).__name__}')
    if df.empty:
        raise ValueError('the profile frame is empty')

    idx = df.index
    if not isinstance(idx, pd.DatetimeIndex):
        raise ValueError(f'index must be a DatetimeIndex, got {type(idx).__name__}')
    if idx.name != 'TIMESTAMP_END':
        raise ValueError(f"index must be named 'TIMESTAMP_END', got {idx.name!r}")
    if idx.tz is not None:
        raise ValueError(f'index must be timezone-naive local time, got tz={idx.tz}')
    if not idx.is_monotonic_increasing:
        raise ValueError('index is not monotonically increasing')
    if idx.has_duplicates:
        raise ValueError(f'index has {int(idx.duplicated().sum())} duplicate timestamp(s)')

    missing = [f for f in fields if f not in df.columns]
    if missing:
        raise ValueError(f'missing column(s): {missing}')

    required = list(fields) if required is None else list(required)
    unknown = [f for f in required if f not in fields]
    if unknown:
        raise ValueError(f'required field(s) not among the requested fields: {unknown}')

    empty = [f for f in fields if df[f].notna().sum() == 0]
    if empty and verbose:
        print(f'(!) REPORT - {len(empty)} of {len(fields)} field(s) hold not a single value in this '
              f'period: {empty}')
    empty_required = [f for f in required if f in empty]
    if empty_required:
        raise ValueError(f'required column(s) without a single value in this period: {empty_required}')
    return df

### Negative control
`check_profile_frame` is the only guard between a broken download and a parquet file that every `TS` screening notebook trusts. A validator that quietly accepts anything is worse than none at all, so prove it rejects each failure mode it claims to catch, accepts a good frame — and, for the split introduced above, prove **both** sides of it: an empty *required* field raises, an empty *non-required* field is accepted and reported.

In [ ]:
_good = pd.DataFrame(
    {f: [7.0, 7.1, 7.2] for f in FIELDS},
    index=pd.DatetimeIndex(pd.date_range('2021-03-26 15:22:00', periods=3, freq='min'),
                           name='TIMESTAMP_END'))
check_profile_frame(_good, FIELDS, required=REQUIRED_FIELDS)  # must NOT raise
print('PASSED - a valid frame is accepted')

_wrong_name = _good.copy()
_wrong_name.index = _good.index.rename('TIMESTAMP_MID')

_tz_aware = _good.copy()
_tz_aware.index = _good.index.tz_localize('UTC')

_empty_required = _good.copy()
_empty_required[REQUIRED_FIELDS[0]] = float('nan')

for _bad, _why in [(_wrong_name, 'index named TIMESTAMP_MID instead of TIMESTAMP_END'),
                   (_tz_aware, 'timezone-aware index'),
                   (_good.drop(columns=[FIELDS[3]]), 'a missing channel column'),
                   (_good.iloc[[0, 0, 1]], 'duplicate timestamps'),
                   (_good.iloc[::-1], 'index not sorted'),
                   (_empty_required, f'an all-NaN REQUIRED field ({REQUIRED_FIELDS[0]})'),
                   (_good.iloc[0:0], 'an empty frame')]:
    try:
        check_profile_frame(_bad, FIELDS, required=REQUIRED_FIELDS)
    except ValueError as e:
        print(f'PASSED - {_why} raises: {e}')
    else:
        raise AssertionError(f'(!) FAILED - check_profile_frame accepted {_why}')

# A REQUIRED_FIELDS list that names something outside FIELDS is a typo, not a relaxation.
try:
    check_profile_frame(_good, FIELDS, required=FIELDS + ['TS_FF1_9.9_9'])
except ValueError as e:
    print(f'PASSED - a required field outside FIELDS raises: {e}')
else:
    raise AssertionError('(!) FAILED - check_profile_frame accepted an unknown required field')

# The other side of the split: an all-NaN NON-required field must be accepted, and reported.
_empty_optional = _good.copy()
_empty_optional['TS_FF1_0.05_3'] = float('nan')
check_profile_frame(_empty_optional, FIELDS, required=REQUIRED_FIELDS)  # must NOT raise
print('PASSED - an all-NaN non-required field is accepted and reported above')

# ... and the strict default (required=None) must still reject exactly that frame, so the
# relaxation is proven to come from REQUIRED_FIELDS and not from a weakened validator.
try:
    check_profile_frame(_empty_optional, FIELDS)
except ValueError as e:
    print(f'PASSED - with the strict default the same frame raises: {e}')
else:
    raise AssertionError('(!) FAILED - the strict default accepted an all-NaN field')

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional — list all fields available in the measurement (does not check the selected time range). Useful here because `TS` also holds the *old* profile's fields (`TS_M5_*` and similar), which belong to different sensors and must not be spliced onto this one:

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download the whole profile
One query, thirteen fields. Returns three objects:

- `data_simple`: the high-res time series, one column per field — **this is what gets saved**.
- `data_detailed`: dict `{field: DataFrame}` with each field's series **and its database tags** — not saved, see *Values only, on purpose* above.
- `assigned_measurements`: the auto-detected measurement per field (a sanity check).

⏳ **This is the slow cell** — about 2.53 million rows over thirteen fields, several minutes. It is the entire reason this notebook exists: run it once here instead of once per screening notebook.

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

## 🔍 Check what came back
Nothing here is optional. The file written below is trusted by every `TS` screening notebook, and each assumption they make about it is asserted here rather than hoped for.

In [ ]:
data_simple

In [ ]:
# All thirteen fields must come from the TS measurement, and nothing else may have slipped
# into the query.
display(assigned_measurements)
assert set(assigned_measurements.keys()) == set(FIELDS), (
    f'(!) unexpected fields returned: {sorted(set(assigned_measurements) ^ set(FIELDS))}')
assert set(assigned_measurements.values()) == {MEASUREMENT}, (
    f'(!) unexpected measurement(s): {sorted(set(assigned_measurements.values()))}')
print(f'OK - all {len(FIELDS)} fields assigned to measurement {MEASUREMENT}')

# The column order of data_simple is the database's, not the order of FIELDS - it puts the two
# late-starting 109 channels last. Readers must select by name; this just makes it visible.
print(f'\nColumn order as returned: {list(data_simple.columns)}')
if list(data_simple.columns) != FIELDS:
    print('    (this differs from the order of FIELDS - expected, and harmless: select by name)')

### Coverage and period
The per-column non-null counts are the quickest way to see that the download is complete and that the holes are the ones the site history explains. Run of **22 Jul 2026** (raw record through 2025): **2 529 202 rows**, first `2020-04-10 15:00:00`, last `2026-01-01 00:00:00`.

| field | records | coverage | first value | min … max [°C] | why it is short |
|---|---:|---:|---|---|---|
| `TS_FF1_0.05_1` | 2 506 674 | 99.11 % | 2020-04-10 15:00 | 0.00 … 41.00 | the shared outages only |
| `TS_FF1_0.05_2` | 2 503 372 | 98.98 % | 2020-04-10 15:00 | 0.00 … 80.00 | shared outages; lost two extra days to the April 2024 bus failure |
| `TS_FF1_0.05_3` | 2 479 116 | 98.02 % | **2021-03-26 15:22** | 1.04 … 20.05 | 109 thermistor, only added at the March 2021 rebuild — 350 days short at the start |
| `TS_FF1_0.05_4` | 2 479 116 | 98.02 % | **2021-03-26 15:22** | **−5.49 … 31.53** | same start; the extreme range is the 2022 fault window, not a colder spot |
| `TS_FF1_0.1_1` | 2 506 841 | 99.12 % | 2020-04-10 15:00 | 0.00 … 40.60 | the shared outages only |
| `TS_FF1_0.1_2` | 2 504 480 | 99.02 % | 2020-04-10 15:00 | 0.00 … 202.50 | shared outages + two extra days in April 2024 |
| `TS_FF1_0.2_1` | 2 491 244 | **98.50 %** | 2020-04-10 15:00 | 0.00 … 53.35 | shared outages **plus the 2025-06-26 … 2025-07-06 intermittent fault** (down to 18-138 records/day) |
| `TS_FF1_0.2_2` | 2 504 897 | 99.04 % | 2020-04-10 15:00 | 0.00 … 121.70 | the shared outages only |
| `TS_FF1_0.3_1` | 2 505 757 | 99.07 % | 2020-04-10 15:00 | 0.00 … 18.40 | the shared outages only |
| `TS_FF1_0.3_2` | **2 058 796** | **81.40 %** | 2020-04-10 15:00 | 0.00 … 18.60 | **the TEROS 12 sensor swap: empty 2024-04-12 … 2025-03-03 (326 days)** |
| `TS_FF1_0.5_1` | 2 505 423 | 99.06 % | 2020-04-10 15:00 | 0.00 … 17.15 | the shared outages only |
| `TS_FF1_0.5_2` | 2 506 194 | 99.09 % | 2020-04-10 15:00 | 0.00 … 277.00 | the shared outages only |
| `TS_FF1_0.6_1` | 2 507 048 | 99.12 % | 2020-04-10 15:00 | 0.00 … 307.56 | the shared outages only |

The "shared outages" are the same five events in every channel — they are power and logger events, not sensor events:

| window | records lost | cause |
|---|---|---|
| 2021-03-25, 03-28, 03-30 … 04-01 | 5 empty days | the FF1 logger box rebuild (fieldbook 2021-03-24/26) |
| 2022-09-07 … 2022-09-15 | 9 empty days, 09-06 and 09-16 partial | the 12 V DC-DC power supply failure, replaced 16 Sep |
| 2023-06-15 | 217 records that day | "no system at ff1 was running for one day" |
| 2024-04-10 … 2024-04-22 | 11-13 empty days, **SDI-12 channels only** | the April 2024 bus failure; the two 109 thermistors logged 1 440/day right through it |
| 2024-06-30 | 1 empty day, 06-29 and 07-01 partial | **no fieldbook entry** — an undocumented site outage that hit all thirteen channels |

Two of these are worth carrying into the screening notebooks. The April 2024 window is the only place where the analogue and the SDI-12 halves of the profile disagree about whether the site was running, which makes `0.05_3`/`0.05_4` the only usable reference across it. And the 2024-06-30 outage is a reminder that the fieldbook is incomplete: a missing entry is not evidence that nothing happened.

Any column falling behind for a reason *not* in these two tables means something went wrong with the download.

In [ ]:
_report = pd.DataFrame({'records': data_simple.count()})
_report['missing'] = len(data_simple) - _report['records']
_report['coverage [%]'] = (_report['records'] / len(data_simple) * 100).round(2)
_report['first'] = [data_simple[c].first_valid_index() for c in _report.index]
_report['last'] = [data_simple[c].last_valid_index() for c in _report.index]
_report['min'] = data_simple.min().round(2)
_report['max'] = data_simple.max().round(2)
_report['mean'] = data_simple.mean().round(2)
_report['required'] = [c in REQUIRED_FIELDS for c in _report.index]
display(_report)

print(f'rows:  {len(data_simple):,}')
print(f'first: {data_simple.index[0]}')
print(f'last:  {data_simple.index[-1]}')
print(f'index: name={data_simple.index.name!r}, dtype={data_simple.index.dtype}, tz={data_simple.index.tz}')

### Validate the frame
The same validator the written file is checked with, applied to what the database returned. If this raises, nothing is written.

In [ ]:
check_profile_frame(data_simple, FIELDS, required=REQUIRED_FIELDS)
print(f'OK - valid profile frame: {len(data_simple):,} rows x {len(data_simple.columns)} channels, '
      f'{data_simple.index[0]} -> {data_simple.index[-1]}')

# The period requested must actually be covered - a silently short download is exactly the kind of
# staleness the screening notebooks would otherwise inherit.
assert data_simple.index[-1] < pd.Timestamp(STOP), '(!) records at or beyond STOP - check the query'
print(f'Period requested: {START} -> {STOP} (stop not included). '
      f'Covered: {data_simple.index[0]} -> {data_simple.index[-1]}')

### The raw record has two time resolutions
The profile is **10MIN until the March 2021 logger rebuild and 1MIN after it** (fieldbook 2021-03-24/26: the FF1 logger box was rebuilt and a new program uploaded; the SDI-12 sensors themselves were only reconnected, not replaced — the two 109 thermistors are the only new hardware, which is why they start here). The database keeps both eras under the same field names, so this single frame has a spacing that changes partway through.

Measured on 22 Jul 2026:

| group | records | share | span | fate in diive |
|---|---:|---:|---|---|
| 60 s | 2 478 898 | 98.01 % | 2021-03-26 15:22 … 2026-01-01 00:00 | kept, the target grid |
| 600 s | 50 079 | 1.98 % | 2020-04-10 15:00 … 2021-03-24 09:50 | kept (above 0.2 %), **upsampled onto the 1MIN grid** |
| 120 s | 5 | 0.00 % | 2024-04-09 11:05 … 2025-11-04 14:16 | dropped (below 0.2 %) |

This is reported rather than fixed. The screening notebooks resample to 30MIN means with a spacing-aware coverage rule, and `StepwiseMeteoScreeningDb` does its own grouping — it drops any frequency group holding less than 0.2 % of the records and upsamples the coarse era onto the fine grid. Both need the raw spacing intact, so this file keeps it.

The consequence for whoever screens these channels: the pre-2021 era arrives as **runs of ten identical values**, so any test built on *differences* degenerates there — a rolling MAD of a mostly-constant series is `0` and the detection band collapses. Report a test's flagged count **per era**, never on the total, and write windows as `60 * 24 * …` so they mean the same wall-clock span in both. Note that `TS_FF1_0.05_3` and `TS_FF1_0.05_4` start *after* the switch and therefore have **only one** resolution era — the per-era split is a no-op for them.

In [ ]:
_freqgroups = detect_freq_groups(index=data_simple.index)
_counts = _freqgroups.value_counts()
print(f'{"freq [s]":>10s} {"records":>10s} {"share":>8s}   first record        last record')
for _f in _counts.index:
    _ix = _freqgroups[_freqgroups == _f].index
    _share = len(_ix) / len(_freqgroups) * 100
    _kept = 'used' if _share > 0.2 else 'dropped by diive (< 0.2%)'
    print(f'{_f:10.0f} {len(_ix):10d} {_share:7.2f}%   {_ix[0]}  {_ix[-1]}   {_kept}')

### Quick look
Daily means of all thirteen channels, as a plausibility check on what is about to be written. Do **not** read quality into this plot — it is the raw record: the `0.0` failed reads and the 2024 decode garbage are in it, and the `0.3_2` line simply stops between April 2024 and March 2025 (and returns from a different probe, 40 cm downslope).

In [ ]:
_daily = data_simple.resample('D').mean()
fig, ax = plt.subplots(figsize=(14, 5))
for _c in data_simple.columns:
    ax.plot(_daily.index, _daily[_c], label=_c, lw=0.9)
ax.set_ylabel('soil temperature [°C]')
ax.set_title('FF1 soil temperature profile, raw daily means (unscreened)')
ax.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.show()

### Does the amplitude still damp with depth?
Reported, not enforced — this notebook screens nothing. But soil temperature has one diagnostic soil moisture does not: the diurnal amplitude must **decrease monotonically with depth**, because the daily wave is damped exponentially as it propagates down. It is the cheapest possible check that the file about to be written contains a physically coherent profile rather than thirteen unrelated series.

Median daily amplitude (daily max − min) over 2023, measured 22 Jul 2026:

| depth | `_1` (water potential sensor) | `_2` (TEROS 12) | 109 thermistors |
|---|---:|---:|---:|
| 0.05 m | 0.40 K | 0.60 K | `_3` 0.73 K, `_4` 1.09 K |
| 0.1 m | 0.30 K | 0.40 K | |
| 0.2 m | 0.20 K | 0.30 K | |
| 0.3 m | 0.20 K | 0.20 K | |
| 0.5 m | 0.10 K | 0.10 K | |
| 0.6 m | 0.10 K | | |

Monotone in both sensor families, and the ordering is preserved. Two caveats before anyone reads more into it. First, the SDI-12 channels are quantised on a 0.05 °C grid, so a 0.10 K amplitude is *two steps* — at 0.5 and 0.6 m the resolution, not the soil, sets the number, and the depths below 0.3 m cannot be separated this way. Second, the 109s are continuous and read higher at the same nominal depth (0.73 and 1.09 K against 0.60 K for the TEROS 12 at 0.05 m); that is partly the missing quantisation and partly a genuinely different placement, and untangling the two is a job for the `0.05_3`/`0.05_4` screening notebooks.

In [ ]:
_yr = data_simple.loc['2023']  # a year with no sensor swap and no long outage
_amp = (_yr.resample('D').max() - _yr.resample('D').min()).median().round(2)
_amp = _amp.rename('median daily amplitude 2023 [K]').to_frame()
_amp['depth [m]'] = [float(c.split('_')[2]) for c in _amp.index]
display(_amp.sort_values(['depth [m]', 'median daily amplitude 2023 [K]']))

# The value grid, which is what separates the two sensor families - see 'Which channel is which'.
_ongrid = {c: round(float(((data_simple[c].dropna() * 20).round() - data_simple[c].dropna() * 20)
                         .abs().lt(1e-6).mean() * 100), 3) for c in data_simple.columns}
print('share of values on a 0.05 degC grid [%]:')
for _c, _v in _ongrid.items():
    print(f'  {_c:16s} {_v:8.3f}   {"SDI-12" if _v > 90 else "analogue (Campbell 109)"}')

## 💾 Export
Write `data_simple` — values only, local `TIMESTAMP_END` — to the external data folder. The file is overwritten on every run, which is what makes re-running after a raw-data extension the whole fix.

In [ ]:
OUTPATH.parent.mkdir(parents=True, exist_ok=True)  # 20_SCREENING writes almost no files, so it may not exist yet
data_simple.to_parquet(OUTPATH)
print(f'Wrote {OUTPATH}')
print(f'      {OUTPATH.stat().st_size / 1024 ** 2:.1f} MB, '
      f'{len(data_simple):,} rows x {len(data_simple.columns)} columns')

### Read the written file back and verify
An export is only as good as what actually landed on disk. Read the parquet back and prove it is the same frame: same shape, same columns in the same order, identical index, identical values (`NaN` included), and valid by the same validator.

In [ ]:
back = pd.read_parquet(OUTPATH)

check_profile_frame(back, FIELDS, required=REQUIRED_FIELDS)
assert back.shape == data_simple.shape, f'(!) shape changed: {back.shape} vs {data_simple.shape}'
assert list(back.columns) == list(data_simple.columns), '(!) column set or order changed'
assert back.index.equals(data_simple.index), '(!) index changed on the round-trip'
pd.testing.assert_frame_equal(back, data_simple)  # values and dtypes, NaN-aware

print('PASSED - the file on disk is identical to what was downloaded.')
print(f'  {len(back):,} rows, {back.index[0]} -> {back.index[-1]} '
      f'(index {back.index.name!r}, tz={back.index.tz})')
display(back.count().rename('records with a value'))

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes for this file

- **What it holds.** The thirteen raw `TS_FF1_*` channels, one float64 column each, on a timezone-naive local (UTC+1) `TIMESTAMP_END` index at the raw spacing (10MIN, then 1MIN from the March 2021 logger rebuild). Soil temperature in **°C**.
- **Column order is the database's, not `FIELDS`'.** The two 109 channels come back last because they start later. Select companions **by name**; never by position.
- **Values only, no tags.** Deliberate — see the section near the top. Companions are used for the 30MIN cross-check, which needs values; the screened variable keeps coming from its own live download, because `StepwiseMeteoScreeningDb` needs the database tags.
- **Raw, so still faulty.** A failed SDI-12 read is `0.00 °C`, not a gap, and `0.00 °C` is a physically possible soil temperature — five such timestamps in this period, listed above. The April 2024 bus failure adds 41 decode-garbage records between 40.5 and 307.56 °C. `TS_FF1_0.3_2` is two probes under one name with a 326-day hole. `TS_FF1_0.05_4` swings ±25 K between 2022-02-24 and 2022-05-12. Filter on a physical range before using any value from this file — and never *clip* to that range, which would turn a failed read into a fabricated number.
- **Not replicates, except at 0.05 m.** The five depths sample different soil; only the four channels at 0.05 m are close to true replicates, and even they are two different sensor types at slightly different placements, so a real offset and a real amplitude difference between them are expected. What *is* physically enforced across the profile is the **ordering**: amplitude damps and the daily maximum lags with depth.
- **Nothing before 2020-04-10 belongs to this profile**, and nothing before 2021-03-26 belongs to `0.05_3`/`0.05_4`. The old forest-floor sensors (Campbell 107, removed 2021-03-25, and the EC-20 moisture profile trashed 2020-03-19) live under different names; splicing them on would join two instruments, not continue one.
- **Addressed by path.** The readers find this file by its full path, exactly like the products in `10_REFERENCE/`, so **renaming or moving it breaks every `TS` screening notebook**. If the period grows, change `OUTNAME`, this notebook's filename, and the readers together.
- **Re-run before the screening notebooks** whenever the raw record is extended or re-ingested. They assert that this file covers the period they screen, so a stale file fails loudly.
- **Not a product.** This is an internal cache of the raw database record, living in `20_SCREENING/` because that is where its readers are. It is not part of the published dataset and nothing downstream of the flux product should read it — the screened, corrected series go to the database and, for the meteo products, to `30_PRODUCTS/`.